PART 03：四个事件，覆盖一次完整的心跳
四个插口各挂在哪个瞬间、能干什么事，一张表说全：
![](四个插口.png)


四个位置卡在一次 agent 心跳的四个关键节点上：输入→执行前→执行后→退出。一个也没多给，一个也没漏。往后不管你想加什么功能，先问一句"它属于哪个节点"，答案必是这四个之一。

挨个挂点真实的东西上去。除了已经搬进来的权限，再挂四个：

日志 hook（PreToolUse）——审计需求落地，两行：

In [ ]:
def log_hook(block):
    args_preview = str(list(block.input.values())[:2])[:60]
    print(f"\033[90m[HOOK] {block.name}({args_preview})\033[0m")
    return None

register_hook("PreToolUse", log_hook)

返回 None：日志只看不管，永远不拦。

自动暂存 hook（PostToolUse）——留痕需求落地：

def auto_git_add_hook(block, output):
    if block.name in ("write_file", "edit_file"):
        path = block.input.get("path", "")
        subprocess.run(f"git add {path}", shell=True, cwd=WORKDIR)
    return None

register_hook("PostToolUse", auto_git_add_hook)
注意它挂在 Post 而不是 Pre：得等文件真的写完，暂存才有意义。选事件就是在选时机，时机错了功能就错了。

大输出警报 hook（PostToolUse）——工具输出超十万字符就亮黄灯，提醒你该给 bash 加截断了：

def large_output_hook(block, output):
    if len(str(output)) > 100000:
        print(f"\033[33m[HOOK] ⚠ {block.name} 输出 {len(str(output))} 字符，注意截断\033[0m")
    return None
会话统计 hook（Stop）——收尾需求落地：

def summary_hook(messages):
    tool_count = sum(
        1 for m in messages
        if isinstance(m.get("content"), list)
        for b in m["content"]
        if isinstance(b, dict) and b.get("type") == "tool_result"
    )
    print(f"\033[90m[HOOK] 本会话共 {tool_count} 次工具调用\033[0m")
    return None

register_hook("Stop", summary_hook)
加一个 UserPromptSubmit 的，每次输入前打一行当前工作目录，让你随时知道它在哪个目录里干活：

def context_hook(query):
    print(f"\033[90m[HOOK] cwd = {WORKDIR}\033[0m")
    return None

register_hook("UserPromptSubmit", context_hook)
到此为止盘点一下：六个功能，循环的改动是零。上一版加一个功能要开一次胸；这一版加一个功能是文件末尾写两行 register_hook——心脏和功能，从此老死不相往来。
![](四个插口执行顺序.png)